![](img/logo_ucm.jpg)

# Model Registry

El componente Model Registry de MLflow proporciona a los desarrolladores una abstracción centralizada para gestionar el ciclo de vida de los modelos. Actúa como un repositorio colaborativo dentro de la organización, permitiendo registrar, versionar, compartir y archivar modelos de manera estructurada. Esta gestión puede realizarse tanto desde las distintas APIs de MLflow como desde su interfaz gráfica.

`````{admonition} Repositorio de modelos
:class: important
El objetivo principal del Model Registry de MLflow es ofrecer un almacén centralizado de modelos donde los equipos de datos puedan almacenar y acceder a versiones relevantes de modelos de machine learning. Una analogía útil sería imaginarlo como un repositorio de Git para modelos, con gestión de versiones, metadatos asociados y estados definidos (como `Staging` o `Production`).
`````

`````{admonition} Cuándo
:class: tip
Registramos un modelo en el Model Registry una vez que hemos finalizado la fase de experimentación y determinado cuál es el modelo óptimo entre los distintos registrados. A partir de ahí, el ciclo de vida del modelo puede ser gestionado formalmente, por ejemplo promoviendo versiones a producción.
`````

Desde la interfaz de usuario de MLflow, una vez finalizada la fase de experimentación, podemos seleccionar el modelo que queremos registrar y poner en producción. El sistema solicitará un nombre para identificar el modelo. Por ejemplo, si el modelo proviene de un experimento con xgboost, podríamos nombrarlo como _iris_model_.


```{figure} img/mlflow/registry2.png
---
name: registry2
align: center
---
Register Model
```


Despues, podemos encontrar  pestaña en el lado derecho de _Experiments_ con la etiqueta _Models_. A través de este módulo, podemos listar todos los modelos registrados, buscar por nombre. Pero además podremos tener siempre una foto de todo el metadatado que hemos ido viendo a lo largo de estas secciones y un control de versiones. 



```{figure} img/mlflow/registry1.png
---
name: registry1
align: center
---
Modelo Resgistrado. Version 1.
```


`````{admonition} Auditoria y Reproducibilidad
:class: tip
Gracias al metadatado almacenado en este modelo con la herramienta de MLflow, podemos siempre volver a carga el modelo y la información básica para revisar métricas, firmas, parámetros, etc. Esto nos ayuda no solo para auditar y mejorar los modelos, sino para reproducir los experimentos.
`````

`````{admonition} ML Engineers
:class: info
En este punto es donde encontraremos todos los modelos de nuestra plataforma. Y también es el punto donde los _data scientist_ y los _ML engineers_ convergen. 
`````

## Model status

Hasta la versión 2.9, MLflow utilizaba un sistema de estados predefinidos para gestionar el ciclo de vida de los modelos registrados. Este sistema indicaba si un modelo se encontraba en fase de pruebas o ya estaba desplegado en producción, mediante las siguientes etiquetas:

- **Staging** (fase de pruebas): Representa un entorno de validación donde el modelo se prueba con datos que simulan el entorno de producción, sin afectar a usuarios finales. Es una etapa clave para garantizar que el modelo funciona correctamente antes de su despliegue.

- **Production** (fase operativa): Es el estado en el que el modelo ya ha sido validado y se encuentra activamente sirviendo predicciones en un entorno real. En esta fase, cualquier fallo puede impactar directamente en usuarios o procesos críticos del negocio.


`````{admonition} Nomenclatura todavía vigente
:class: warning
Aunque nos encontramos que MLflow ha dejado de trabajar con esta terminología, en muchos escenarios productivos todavía se sigue utilizando esta nomenclatura.
`````

### Alias en el Model Registry

En las versiones más recientes de MLflow, se introducen los **alias**, una funcionalidad que permite asignar etiquetas simbólicas a versiones específicas de modelos registrados. Estos alias proporcionan una forma flexible y conveniente de referirse a versiones concretas, facilitando su gestión y despliegue en distintos entornos de trabajo.

Un alias es simplemente un nombre simbólico que apunta a una versión específica de un modelo en el registro de modelos de MLflow. Por ejemplo, puedes asignar el alias `"champion"` a la versión del modelo que actualmente está en producción.

#### Beneficios de usar alias

- **Flexibilidad en el despliegue**: Los alias permiten desacoplar el proceso de despliegue del código de inferencia. Puedes reasignar un alias a una nueva versión de modelo sin necesidad de modificar el código que lo consume. Esto facilita actualizaciones y pruebas A/B.

- **Facilidad de uso**: Los alias simplifican la referencia a versiones de modelos, permitiendo usar nombres simbólicos como `"champion"` o `"candidate"` en lugar de manejar números de versión específicos.

- **Mejor gestión de versiones**: Facilitan una organización más clara del ciclo de vida del modelo. Es posible mantener múltiples alias con diferentes propósitos (por ejemplo, `"champion"` para producción, `"staging"` para pruebas), y reasignarlos según sea necesario.


```{figure} img/mlflow/registry3.png
---
name: registry3
align: center
---
Alias.
```



`````{admonition} Registro y despliegue del modelo 
:class: info
Una vez registrado el modelo, querremos consumirlo como ya hemos visto en anteriores secciones con FastAPI. MLflow puede hacerlo de manera sencilla con su componente **Model Serving** que veremos en la siguiente sección. 
`````

In [12]:
from mlflow import MlflowClient
import time

# Crea una URL unica en https://webhook.site y pegala aqui.
# WEBHOOK_SITE_URL = "https://webhook.site/REEMPLAZAR_POR_TU_UUID"
WEBHOOK_SITE_URL = "https://webhook.site/40138614-8d87-42fe-a8e7-b7ff6e0cbc3d"

client = MlflowClient(tracking_uri="http://127.0.0.1:5000")

# Limpia webhooks previos con el mismo nombre/destino para evitar duplicados.
for existing in client.list_webhooks(max_results=100):
    if existing.name == "alias-change-hook-webhook-site" and existing.url == WEBHOOK_SITE_URL:
        client.delete_webhook(existing.webhook_id)

webhook = client.create_webhook(
    name="alias-change-hook-webhook-site",
    url=WEBHOOK_SITE_URL,
    events=[
        "model_version_alias.created",
        "model_version_alias.deleted",
    ],
    description="Prueba de webhook para cambios de alias en Model Registry",
)

time.sleep(1)
print("Webhook creado:", webhook.webhook_id)
print("Destino:", webhook.url)
print("Abre la URL en Webhook.site para inspeccionar los POST recibidos.")

Webhook creado: 48cce7bf-1e88-405a-bcef-55ca49926d06
Destino: https://webhook.site/40138614-8d87-42fe-a8e7-b7ff6e0cbc3d
Abre la URL en Webhook.site para inspeccionar los POST recibidos.


In [15]:
# SqlAlchemyStore permite crear webhooks, pero no implementa client.test_webhook(...).
# Para probarlo en local hay que lanzar un evento real del registry.
import time 
MODEL_NAME = "mynewmodel"
ALIAS_NAME = "champion"
TARGET_VERSION = "1"

# Fuerza primero un borrado para provocar un evento 'deleted' si el alias ya existia.
try:
    client.delete_registered_model_alias(name=MODEL_NAME, alias=ALIAS_NAME)
    print(f"Alias {ALIAS_NAME!r} eliminado antes de recrearlo")
    time.sleep(1)
except Exception:
    print(f"Alias {ALIAS_NAME!r} no existia; se crea directamente")

client.set_registered_model_alias(
    name=MODEL_NAME,
    alias=ALIAS_NAME,
    version=TARGET_VERSION,
)
time.sleep(2)

assigned = client.get_model_version_by_alias(
    name=MODEL_NAME,
    alias=ALIAS_NAME,
)
print(f"Alias {ALIAS_NAME!r} asignado a {assigned.name} v{assigned.version}")
print("Revisa ahora webhook.site: deberias ver un POST con action='deleted' y otro con action='created'.")


Alias 'champion' eliminado antes de recrearlo
Alias 'champion' asignado a mynewmodel v1
Revisa ahora webhook.site: deberias ver un POST con action='deleted' y otro con action='created'.


In [ ]:
https://webhook.site/#!/view/40138614-8d87-42fe-a8e7-b7ff6e0cbc3d/26d26b5f-edcf-4c31-89b3-8d5cc571e08d/1